# DecompDiff local sweep 2 — loss / prediction ablation

Trains the **trend + season + residual → fusion DiT** DecompDiff and logs every run to
**wandb** (project `decompdiff-sweep2`, group `residual-fusion-sweep`).

Fixed architecture for all runs: `num_layers=1`, `num_fusion_layers=1`, `hidden_dim=64`,
window 32, 2000 epochs. Only the **training objective** varies:

| # | loss | prediction | loss weight |
|---|------|-----------|:-----------:|
| 1 | L1   | x0        | on (Diffusion-TS) — baseline |
| 2 | MSE  | x0        | on |
| 3 | L1   | x0        | off |
| 4 | MSE  | x0        | off |
| 5 | L1   | eps (noise) | off |
| 6 | MSE  | eps (noise) | off |

Each cell is an independent wandb run — run them one at a time. Compare runs by
disc / pred / vds / fdds (train-loss values are **not** comparable across loss/prediction modes).


In [1]:
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from pathlib import Path
import wandb

REPO = Path("C:/Users/ameli/Desktop/TezBaselines")
sys.path.insert(0, str(REPO))

from DecompDiff.models.decompDiff import DecompDiff
from DecompDiff.models.diffusion  import GaussianDiffusion
from DecompDiff.config.stocks_config import Config
from MyCode.utils.data_utils.loader import create_data_loaders
from MyCode.eval_metrics import evaluate_samples, vds_score, fdds_score, correlational_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


device: cuda


In [2]:
def compute_loss(model, diffusion, x_0, loss_type="l1",
                 prediction_type="x0", use_loss_weight=True):
    # prediction_type: "x0" -> target is clean sample, "eps" -> target is the noise
    # use_loss_weight: Diffusion-TS per-timestep weight; set False for plain loss
    B = x_0.shape[0]
    t = torch.randint(0, diffusion.num_timesteps, (B,), device=x_0.device)
    x_t, noise = diffusion.q_sample(x_0, t)
    model_out = model(x_t, t)
    target = x_0 if prediction_type == "x0" else noise
    loss = F.l1_loss(model_out, target, reduction="none") if loss_type == "l1" \
           else F.mse_loss(model_out, target, reduction="none")
    loss = loss.mean(dim=[1, 2])                       # (B,)
    if use_loss_weight:
        loss = loss * diffusion.loss_weight[t]
    return loss.mean()


def compute_inline_metrics(model, diffusion, real_loader, device, num_steps=50,
                           prediction_type="x0"):
    # DiffusionTS protocol: ALL real windows, fake count = N, no cap.
    model.eval()

    batches = []
    for b in real_loader:
        batches.append(b.cpu().numpy())
    real_CL = np.concatenate(batches, axis=0)   # (N, C, L)
    N = real_CL.shape[0]

    chunk = 256
    chunks = []
    with torch.no_grad():
        for start in range(0, N, chunk):
            bs = min(chunk, N - start)
            chunks.append(
                model.sample(diffusion, batch_size=bs, num_steps=num_steps, eta=0.0,
                             prediction_type=prediction_type).cpu()
            )
    fake_CL = torch.cat(chunks, dim=0).numpy()  # (N, C, L)

    real_m = ((real_CL.transpose(0, 2, 1) + 1.0) * 0.5).astype(np.float32)
    fake_m = ((fake_CL.transpose(0, 2, 1) + 1.0) * 0.5).astype(np.float32)

    try:
        results = evaluate_samples(real_m, fake_m, device=device,
                                   n_iterations=3,
                                   disc_iterations=2000,
                                   pred_iterations=5000)
        vds  = vds_score(real_m, fake_m)
        fdds = fdds_score(real_m, fake_m)
        corr = correlational_score(real_m, fake_m)
    except Exception as e:
        print(f"  [metrics] failed: {e}")
        model.train()
        return None

    model.train()
    return {
        "disc_score":          results["discriminative"]["mean"],
        "disc_score_std":      results["discriminative"]["std"],
        "test_acc":            results["discriminative"]["test_acc"],
        "pred_mae":            results["predictive"]["mean"],
        "pred_mae_std":        results["predictive"]["std"],
        "vds":                 vds,
        "fdds":                fdds,
        "correlational_score": corr,
    }


In [ ]:
def run_experiment(window_length, num_epochs, num_layers=None, num_fusion_layers=None,
                   hidden_dim=None, eval_every=500, run_name=None,
                   prediction_type="x0", use_loss_weight=True, loss_type="l1",
                   dataset="stock"):
    """Train one DecompDiff (residual-fusion) config and log it to wandb.

    hidden_dim == 0  ->  no projection: model runs on the raw input channels.
    prediction_type : "x0" (predict clean sample) or "eps" (predict noise).
    use_loss_weight : True = Diffusion-TS per-timestep loss weight, False = plain loss.
    loss_type       : "l1" or "mse".
    dataset         : stock, etth1, etth2, exchange, fmri, eeg, sine
                      (input_channels is set automatically from the dataset).
    """
    from DecompDiff.data.datasets import make_loaders

    cfg = Config()
    cfg.model.sequence_length = window_length
    cfg.training.num_epochs   = num_epochs
    cfg.data.dataset          = dataset
    if num_layers        is not None: cfg.model.num_layers        = num_layers
    if num_fusion_layers is not None: cfg.model.num_fusion_layers = num_fusion_layers
    if hidden_dim        is not None: cfg.model.hidden_dim        = hidden_dim
    cfg.training.prediction_type = prediction_type
    cfg.training.use_loss_weight = use_loss_weight
    cfg.training.loss_type       = loss_type

    # ── data (any dataset) — input_channels is set from what the data provides ──
    train_loader, _, ds = make_loaders(
        dataset,
        batch_size     = cfg.training.batch_size,
        window         = window_length,
        train_ratio    = cfg.data.train_split,
        neg_one_to_one = cfg.data.neg_one_to_one,
        per_window     = cfg.data.per_window_norm,
        num_workers    = cfg.data.num_workers,
        pin_memory     = False,
        data_root      = cfg.data.data_root,
        sine_num       = cfg.data.sine_num,
        sine_dim       = cfg.data.sine_dim,
        seed           = cfg.data.sine_seed,
    )
    cfg.model.input_channels = ds.num_features

    h    = cfg.model.hidden_dim
    htag = f"H{h}" if h > 0 else "H0-noproj"
    wtag = "w1" if use_loss_weight else "w0"
    run_name = run_name or (
        f"decompdiff-{dataset}-L{window_length}-{htag}"
        f"-NL{cfg.model.num_layers}-NF{cfg.model.num_fusion_layers}"
        f"-{loss_type}-{prediction_type}-{wtag}-E{num_epochs}"
    )

    wandb.init(
        project = "decompdiff-sweep2",
        group   = "residual-fusion-sweep",
        name    = run_name,
        tags    = [dataset, "proj" if h > 0 else "noproj",
                   f"NL{cfg.model.num_layers}", f"NF{cfg.model.num_fusion_layers}",
                   f"pred-{prediction_type}", wtag, f"loss-{loss_type}"],
        config  = {
            **cfg.model.__dict__,
            **cfg.diffusion.__dict__,
            **cfg.training.__dict__,
            "dataset":    dataset,
            "device":     DEVICE,
            "eval_every": eval_every,
            "projection": h > 0,
        },
    )
    print(f"[{run_name}] train batches: {len(train_loader)}  "
          f"channels={cfg.model.input_channels}  (train_split={cfg.data.train_split})")

    model = DecompDiff(
        input_channels    = cfg.model.input_channels,
        sequence_length   = window_length,
        hidden_dim        = cfg.model.hidden_dim,
        num_heads         = cfg.model.num_heads,
        num_layers        = cfg.model.num_layers,
        num_fusion_layers = cfg.model.num_fusion_layers,
        mlp_ratio         = cfg.model.mlp_ratio,
        dropout           = cfg.model.dropout,
        freq_dim          = cfg.model.freq_dim,
    ).to(DEVICE)

    diffusion = GaussianDiffusion(
        num_timesteps  = cfg.diffusion.num_timesteps,
        beta_start     = cfg.diffusion.beta_start,
        beta_end       = cfg.diffusion.beta_end,
        noise_schedule = cfg.diffusion.noise_schedule,
        device         = DEVICE,
    ).to(DEVICE)

    counts = model.get_parameter_count()
    print(f"[{run_name}] model_dim={model.model_dim}  params: {counts['total']:,}  "
          f"loss={loss_type}  pred={prediction_type}  loss_weight={use_loss_weight}")
    wandb.config.update({"total_params": counts["total"], "model_dim": model.model_dim},
                        allow_val_change=True)

    total_steps = len(train_loader) * num_epochs
    optimizer = AdamW(model.parameters(), lr=cfg.training.learning_rate,
                      weight_decay=cfg.training.weight_decay, betas=(0.9, 0.999))
    warmup = LinearLR(optimizer, start_factor=1e-3, end_factor=1.0,
                      total_iters=cfg.training.warmup_steps)
    cosine = CosineAnnealingLR(optimizer,
                               T_max=max(1, total_steps - cfg.training.warmup_steps),
                               eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine],
                              milestones=[cfg.training.warmup_steps])

    ckpt_dir = REPO / f"DecompDiff/output/checkpoints/{run_name}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    save_every = max(100, num_epochs // 5)

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        for batch in train_loader:
            x_0 = batch.to(DEVICE)
            optimizer.zero_grad()
            loss = compute_loss(model, diffusion, x_0, loss_type,
                                prediction_type=prediction_type,
                                use_loss_weight=use_loss_weight)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.training.gradient_clip_val)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

        train_loss = epoch_loss / len(train_loader)
        current_lr = optimizer.param_groups[0]["lr"]
        log = {"train_loss": train_loss, "lr": current_lr, "epoch": epoch + 1}

        if (epoch + 1) % save_every == 0 or (epoch + 1) == num_epochs:
            torch.save({
                "epoch":            epoch + 1,
                "train_loss":       train_loss,
                "model_state_dict": model.state_dict(),
                "config":           {"model": cfg.model.__dict__,
                                     "window_length": window_length,
                                     "dataset": dataset,
                                     "prediction_type": prediction_type,
                                     "use_loss_weight": use_loss_weight,
                                     "loss_type": loss_type},
            }, ckpt_dir / f"checkpoint_ep{epoch+1}.pt")

        if (epoch + 1) % eval_every == 0:
            print(f"  [epoch {epoch+1}] computing metrics (all windows)...")
            metrics = compute_inline_metrics(model, diffusion, train_loader, DEVICE,
                                             num_steps=50, prediction_type=prediction_type)
            if metrics is not None:
                log.update(metrics)
                print(
                    f"  disc={metrics['disc_score']:.4f} "
                    f"pred={metrics['pred_mae']:.4f} "
                    f"vds={metrics['vds']:.4f} "
                    f"fdds={metrics['fdds']:.4f} "
                    f"corr={metrics['correlational_score']:.4f}"
                )

        wandb.log(log)
        print(f"[{run_name}] ep {epoch+1:4d}/{num_epochs}  "
              f"train={train_loss:.5f}  lr={current_lr:.2e}")

    print(f"[{run_name}] done.  ckpt -> {ckpt_dir}")
    wandb.finish()
    return model, diffusion


In [15]:
# ── shared settings for the whole sweep ──────────────────────────────────────
WINDOW     = 32
EPOCHS     = 2000
EVAL_EVERY = 250


## Loss / prediction ablation — fixed architecture NL1 · NF1 · hidden_dim 64

All six runs use the **same model** (channels `num_layers=1`, fusion `num_fusion_layers=1`,
`hidden_dim=64`, window 32, 2000 epochs) and vary only the training objective:

| # | loss | prediction | loss weight |
|---|------|-----------|:-----------:|
| 1 | L1   | x0        | on (Diffusion-TS) |
| 2 | MSE  | x0        | on |
| 3 | L1   | x0        | off |
| 4 | MSE  | x0        | off |
| 5 | L1   | eps (noise) | off |
| 6 | MSE  | eps (noise) | off |

Run one cell at a time — each is an independent wandb run.


In [16]:
# ── Exp 1: baseline  ·  L1 · predict x0 · loss weight ON ─────────────────────
run_experiment(window_length=WINDOW, num_epochs=EPOCHS, eval_every=EVAL_EVERY,
               num_layers=1, num_fusion_layers=1, hidden_dim=64,
               loss_type="l1", prediction_type="x0", use_loss_weight=True)


StockDataset: 3654 windows  (train=3654, test=all [train_ratio=1.0])
[decompdiff-L32-H64-NL1-NF1-l1-x0-w1-E2000] train batches: 58  (train_split=1.0)
[decompdiff-L32-H64-NL1-NF1-l1-x0-w1-E2000] model_dim=64  params: 252,550  loss=l1  pred=x0  loss_weight=True
[decompdiff-L32-H64-NL1-NF1-l1-x0-w1-E2000] ep    1/2000  train=1.09652  lr=5.80e-05
[decompdiff-L32-H64-NL1-NF1-l1-x0-w1-E2000] ep    2/2000  train=0.71845  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-x0-w1-E2000] ep    3/2000  train=0.28758  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-x0-w1-E2000] ep    4/2000  train=0.18442  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-x0-w1-E2000] ep    5/2000  train=0.16242  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-x0-w1-E2000] ep    6/2000  train=0.15460  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-x0-w1-E2000] ep    7/2000  train=0.14864  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-x0-w1-E2000] ep    8/2000  train=0.14324  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-x0-w1-E2000] ep    9/2000  tr

correlational_score,▇▃█▁▆▇▇▄
disc_score,▆▁▄▅█▄▅▁
disc_score_std,▁█▃▄▄▂▇▃
epoch,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▆▆▆▆▇█████
fdds,▁█▇█▅▅▇▅
lr,█████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁
pred_mae,▄▁▇▂█▆▇▃
pred_mae_std,▃▇▃▃▃█▆▁
test_acc,▆▁▄▅█▄▅▁
train_loss,█▅▆▄▃▃▃▄▃▃▂▃▃▂▂▂▂▁▂▃▃▂▃▂▁▂▂▂▂▃▂▁▁▁▁▁▂▂▁▃
+1,...


(DecompDiff(
   (decomp): SeriesDecomposition()
   (time_embedder): TimestepEmbedder(
     (sinusoidal): SinusoidalEmbedding()
     (mlp): Sequential(
       (0): Linear(in_features=256, out_features=64, bias=True)
       (1): SiLU()
       (2): Linear(in_features=64, out_features=64, bias=True)
     )
   )
   (trend_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (season_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (res_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (trend_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (season_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (res_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (trend_dit): DiTStack(
     (blocks): ModuleList(
       (0): DiTBlock(
         (norm1): LayerNorm((64,), eps=1e-06, elementwise_affine=False)
         (norm2): LayerNorm((64,), eps=1e-

In [17]:
# ── Exp 2: MSE · predict x0 · loss weight ON ─────────────────────────────────
run_experiment(window_length=WINDOW, num_epochs=EPOCHS, eval_every=EVAL_EVERY,
               num_layers=1, num_fusion_layers=1, hidden_dim=64,
               loss_type="mse", prediction_type="x0", use_loss_weight=True)


StockDataset: 3654 windows  (train=3654, test=all [train_ratio=1.0])
[decompdiff-L32-H64-NL1-NF1-mse-x0-w1-E2000] train batches: 58  (train_split=1.0)
[decompdiff-L32-H64-NL1-NF1-mse-x0-w1-E2000] model_dim=64  params: 252,550  loss=mse  pred=x0  loss_weight=True
[decompdiff-L32-H64-NL1-NF1-mse-x0-w1-E2000] ep    1/2000  train=0.76953  lr=5.80e-05
[decompdiff-L32-H64-NL1-NF1-mse-x0-w1-E2000] ep    2/2000  train=0.39342  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-x0-w1-E2000] ep    3/2000  train=0.07610  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-x0-w1-E2000] ep    4/2000  train=0.04385  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-x0-w1-E2000] ep    5/2000  train=0.03645  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-x0-w1-E2000] ep    6/2000  train=0.03254  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-x0-w1-E2000] ep    7/2000  train=0.03054  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-x0-w1-E2000] ep    8/2000  train=0.02944  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-x0-w1-E2000] ep  

correlational_score,█▇▁▁▄▁▄▂
disc_score,▄█▃▄▂▂▁▁
disc_score_std,▃█▃▃▂▁▁▁
epoch,▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
fdds,▁▃▆▇▆█▇▆
lr,█████████▇▇▇▇▇▆▆▅▅▅▅▅▅▅▄▄▄▄▄▃▃▂▁▁▁▁▁▁▁▁▁
pred_mae,▃█▂▂▁▁▁▁
pred_mae_std,▇█▃▇▇▃▅▁
test_acc,▄█▃▄▁▂▁▁
train_loss,█▅▆▃▃▃▂▂▆▄▅▅▁▄▃▃▃▃▁▃▃▁▂▄▄▄▁▃▁▄▅▂▃▃▁▁▂▃▂▃
+1,...


(DecompDiff(
   (decomp): SeriesDecomposition()
   (time_embedder): TimestepEmbedder(
     (sinusoidal): SinusoidalEmbedding()
     (mlp): Sequential(
       (0): Linear(in_features=256, out_features=64, bias=True)
       (1): SiLU()
       (2): Linear(in_features=64, out_features=64, bias=True)
     )
   )
   (trend_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (season_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (res_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (trend_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (season_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (res_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (trend_dit): DiTStack(
     (blocks): ModuleList(
       (0): DiTBlock(
         (norm1): LayerNorm((64,), eps=1e-06, elementwise_affine=False)
         (norm2): LayerNorm((64,), eps=1e-

In [18]:
# ── Exp 3: L1 · predict x0 · loss weight OFF ─────────────────────────────────
run_experiment(window_length=WINDOW, num_epochs=EPOCHS, eval_every=EVAL_EVERY,
               num_layers=1, num_fusion_layers=1, hidden_dim=64,
               loss_type="l1", prediction_type="x0", use_loss_weight=False)


StockDataset: 3654 windows  (train=3654, test=all [train_ratio=1.0])
[decompdiff-L32-H64-NL1-NF1-l1-x0-w0-E2000] train batches: 58  (train_split=1.0)
[decompdiff-L32-H64-NL1-NF1-l1-x0-w0-E2000] model_dim=64  params: 252,550  loss=l1  pred=x0  loss_weight=False
[decompdiff-L32-H64-NL1-NF1-l1-x0-w0-E2000] ep    1/2000  train=0.75989  lr=5.80e-05
[decompdiff-L32-H64-NL1-NF1-l1-x0-w0-E2000] ep    2/2000  train=0.49948  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-x0-w0-E2000] ep    3/2000  train=0.19244  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-x0-w0-E2000] ep    4/2000  train=0.14096  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-x0-w0-E2000] ep    5/2000  train=0.12777  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-x0-w0-E2000] ep    6/2000  train=0.12150  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-x0-w0-E2000] ep    7/2000  train=0.11512  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-x0-w0-E2000] ep    8/2000  train=0.11088  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-x0-w0-E2000] ep    9/2000  t

correlational_score,█▄▃▁▁▃▄▅
disc_score,▅▁▆█▆▆▆▇
disc_score_std,▇▆▁▄▆▅█▂
epoch,▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇████
fdds,▁▃▃▆▃█▇█
lr,███████▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
pred_mae,▄▁▆█▇▅▇█
pred_mae_std,▂▁▂▅▂▃█▄
test_acc,▅▁▆█▆▆▆▇
train_loss,█▅▇▆▅▃▃▆▅▃▅▆▂▃▇▃▄▅▄▄▄▂▄▄▄▃▄▂▃▄▃▅▄▄▄▅▃▁▃▄
+1,...


(DecompDiff(
   (decomp): SeriesDecomposition()
   (time_embedder): TimestepEmbedder(
     (sinusoidal): SinusoidalEmbedding()
     (mlp): Sequential(
       (0): Linear(in_features=256, out_features=64, bias=True)
       (1): SiLU()
       (2): Linear(in_features=64, out_features=64, bias=True)
     )
   )
   (trend_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (season_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (res_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (trend_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (season_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (res_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (trend_dit): DiTStack(
     (blocks): ModuleList(
       (0): DiTBlock(
         (norm1): LayerNorm((64,), eps=1e-06, elementwise_affine=False)
         (norm2): LayerNorm((64,), eps=1e-

In [19]:
# ── Exp 4: MSE · predict x0 · loss weight OFF ────────────────────────────────
run_experiment(window_length=WINDOW, num_epochs=EPOCHS, eval_every=EVAL_EVERY,
               num_layers=1, num_fusion_layers=1, hidden_dim=64,
               loss_type="mse", prediction_type="x0", use_loss_weight=False)


StockDataset: 3654 windows  (train=3654, test=all [train_ratio=1.0])
[decompdiff-L32-H64-NL1-NF1-mse-x0-w0-E2000] train batches: 58  (train_split=1.0)
[decompdiff-L32-H64-NL1-NF1-mse-x0-w0-E2000] model_dim=64  params: 252,550  loss=mse  pred=x0  loss_weight=False
[decompdiff-L32-H64-NL1-NF1-mse-x0-w0-E2000] ep    1/2000  train=0.54019  lr=5.80e-05
[decompdiff-L32-H64-NL1-NF1-mse-x0-w0-E2000] ep    2/2000  train=0.28771  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-x0-w0-E2000] ep    3/2000  train=0.06652  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-x0-w0-E2000] ep    4/2000  train=0.05085  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-x0-w0-E2000] ep    5/2000  train=0.04323  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-x0-w0-E2000] ep    6/2000  train=0.04194  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-x0-w0-E2000] ep    7/2000  train=0.03777  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-x0-w0-E2000] ep    8/2000  train=0.03720  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-x0-w0-E2000] ep 

correlational_score,▂▇█▃▅▁▄▃
disc_score,▂█▅▁▁▂▁▁
disc_score_std,▂█▁▁▁▁▁▁
epoch,▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇███
fdds,▅▁▁▄▄█▇█
lr,██▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
pred_mae,▅█▄▂▁▃▁▁
pred_mae_std,▄▃█▅▂▁▂▁
test_acc,▂█▅▁▁▂▁▁
train_loss,█▂▃▄▃▅▂▃▆▃▃▄▃▃▂▃▂▃▂▄▁▆▃▅▃▄▃▃▄▄▃▂▃▄▃▃▃▁▁▆
+1,...


(DecompDiff(
   (decomp): SeriesDecomposition()
   (time_embedder): TimestepEmbedder(
     (sinusoidal): SinusoidalEmbedding()
     (mlp): Sequential(
       (0): Linear(in_features=256, out_features=64, bias=True)
       (1): SiLU()
       (2): Linear(in_features=64, out_features=64, bias=True)
     )
   )
   (trend_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (season_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (res_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (trend_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (season_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (res_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (trend_dit): DiTStack(
     (blocks): ModuleList(
       (0): DiTBlock(
         (norm1): LayerNorm((64,), eps=1e-06, elementwise_affine=False)
         (norm2): LayerNorm((64,), eps=1e-

In [20]:
# ── Exp 5: L1 · predict eps (noise) · loss weight OFF ────────────────────────
run_experiment(window_length=WINDOW, num_epochs=EPOCHS, eval_every=EVAL_EVERY,
               num_layers=1, num_fusion_layers=1, hidden_dim=64,
               loss_type="l1", prediction_type="eps", use_loss_weight=False)


StockDataset: 3654 windows  (train=3654, test=all [train_ratio=1.0])
[decompdiff-L32-H64-NL1-NF1-l1-eps-w0-E2000] train batches: 58  (train_split=1.0)
[decompdiff-L32-H64-NL1-NF1-l1-eps-w0-E2000] model_dim=64  params: 252,550  loss=l1  pred=eps  loss_weight=False
[decompdiff-L32-H64-NL1-NF1-l1-eps-w0-E2000] ep    1/2000  train=0.90093  lr=5.80e-05
[decompdiff-L32-H64-NL1-NF1-l1-eps-w0-E2000] ep    2/2000  train=0.76361  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-eps-w0-E2000] ep    3/2000  train=0.38484  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-eps-w0-E2000] ep    4/2000  train=0.22558  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-eps-w0-E2000] ep    5/2000  train=0.17697  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-eps-w0-E2000] ep    6/2000  train=0.15222  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-eps-w0-E2000] ep    7/2000  train=0.13783  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-eps-w0-E2000] ep    8/2000  train=0.12911  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-l1-eps-w0-E2000] ep 

correlational_score,█▁▂▃▄▁▄▅
disc_score,█▄▁▃▂▂▃▄
disc_score_std,▃▂▁█▁▃▁▂
epoch,▁▁▁▁▁▁▁▂▂▂▂▂▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▇▇▇▇▇▇▇█████
fdds,▁▃▃▂▇▄▇█
lr,█████▇▇▇▇▆▆▆▆▆▆▅▅▅▅▄▄▄▄▄▄▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁
pred_mae,█▁▁▂▁▁▂▂
pred_mae_std,▅▃▁▂▇▅▅█
test_acc,█▄▁▃▂▂▃▄
train_loss,█▄▃▃▄▃▃▃▄▄▃▃▃▃▂▃▁▂▂▂▂▂▂▂▂▂▁▂▁▂▁▂▂▁▂▂▁▂▂▁
+1,...


(DecompDiff(
   (decomp): SeriesDecomposition()
   (time_embedder): TimestepEmbedder(
     (sinusoidal): SinusoidalEmbedding()
     (mlp): Sequential(
       (0): Linear(in_features=256, out_features=64, bias=True)
       (1): SiLU()
       (2): Linear(in_features=64, out_features=64, bias=True)
     )
   )
   (trend_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (season_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (res_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (trend_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (season_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (res_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (trend_dit): DiTStack(
     (blocks): ModuleList(
       (0): DiTBlock(
         (norm1): LayerNorm((64,), eps=1e-06, elementwise_affine=False)
         (norm2): LayerNorm((64,), eps=1e-

In [21]:
# ── Exp 6: MSE · predict eps (noise) · loss weight OFF ───────────────────────
run_experiment(window_length=WINDOW, num_epochs=EPOCHS, eval_every=EVAL_EVERY,
               num_layers=1, num_fusion_layers=1, hidden_dim=64,
               loss_type="mse", prediction_type="eps", use_loss_weight=False)


StockDataset: 3654 windows  (train=3654, test=all [train_ratio=1.0])
[decompdiff-L32-H64-NL1-NF1-mse-eps-w0-E2000] train batches: 58  (train_split=1.0)
[decompdiff-L32-H64-NL1-NF1-mse-eps-w0-E2000] model_dim=64  params: 252,550  loss=mse  pred=eps  loss_weight=False
[decompdiff-L32-H64-NL1-NF1-mse-eps-w0-E2000] ep    1/2000  train=1.04003  lr=5.80e-05
[decompdiff-L32-H64-NL1-NF1-mse-eps-w0-E2000] ep    2/2000  train=0.77939  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-eps-w0-E2000] ep    3/2000  train=0.22619  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-eps-w0-E2000] ep    4/2000  train=0.11965  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-eps-w0-E2000] ep    5/2000  train=0.08247  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-eps-w0-E2000] ep    6/2000  train=0.06804  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-eps-w0-E2000] ep    7/2000  train=0.05753  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-eps-w0-E2000] ep    8/2000  train=0.05162  lr=1.00e-04
[decompdiff-L32-H64-NL1-NF1-mse-eps-w

correlational_score,▄█▁▂▄▂▁▂
disc_score,█▄▁▁▃▂▁▁
disc_score_std,▅▄▄▃█▂▅▁
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇████
fdds,▁▆▂▂▃█▆▆
lr,██████████▇▇▇▇▇▆▅▄▄▄▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁
pred_mae,▃▄▁▁█▂▁▁
pred_mae_std,█▄▂▁▆▄▆▁
test_acc,█▄▁▁▃▂▁▂
train_loss,█▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+1,...


(DecompDiff(
   (decomp): SeriesDecomposition()
   (time_embedder): TimestepEmbedder(
     (sinusoidal): SinusoidalEmbedding()
     (mlp): Sequential(
       (0): Linear(in_features=256, out_features=64, bias=True)
       (1): SiLU()
       (2): Linear(in_features=64, out_features=64, bias=True)
     )
   )
   (trend_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (season_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (res_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (trend_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (season_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (res_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (trend_dit): DiTStack(
     (blocks): ModuleList(
       (0): DiTBlock(
         (norm1): LayerNorm((64,), eps=1e-06, elementwise_affine=False)
         (norm2): LayerNorm((64,), eps=1e-